In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [2]:
model_name = "scb10x/llama3.2-typhoon2-1b-instruct"

model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"}
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    padding=True,
    return_tensors="pt"
)

outputs = model.generate(
                input_ids,
                max_new_tokens=512,
                eos_token_id=tokenizer.eos_token_id,
                do_sample=True,
                temperature=0.9,
                top_p=0.9,
                top_k=64
        )
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

[2025-05-15 09:44:54,014] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to mps (auto detect)


W0515 09:44:54.178000 34238 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is the capital of France?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The capital of France is Paris.<|eot_id|>


In [2]:
# Import dataset
import json
import os

def get_data_from_json(file_path):
    """Load data from a JSON file"""
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

base_path = os.getcwd()
# Load data from JSON file
training_data = get_data_from_json(os.path.join(base_path, "data/klon8_data/non_instruct_tagged_train_pretrain.json"))
new_token = get_data_from_json(os.path.join(base_path, "data/klon8_data/phonetic_token.json"))


In [3]:
tmp_data =get_data_from_json(os.path.join(base_path, "data/klon8_data/instruct_untagged_train_sft.json"))

len(tmp_data)

9478

In [4]:
import numpy as np

# Calculate steps
total_train_samples = len(tmp_data)
max_samples = 1000
total_train_samples = min(total_train_samples, max_samples)
batch_size = 8
accumulation_steps = 2
epochs = 2

save_steps_per_epoch = 1
eval_times_per_epoch = 1
every_n_steps_per_epoch = 10

total_steps = int(np.ceil(total_train_samples / (batch_size * accumulation_steps)) * epochs)
every_n_steps = total_steps // (epochs * every_n_steps_per_epoch)
warmup_steps = int(0.1 * total_steps) # 10% warmup
# Evaluate and save based on save_steps_per_epoch
save_steps = int(np.ceil(total_steps / (save_steps_per_epoch * epochs)))
eval_steps = total_steps // (epochs * eval_times_per_epoch)

# # Calculate eval_steps to ensure it divides save_steps evenly
# ideal_eval_frequency = total_steps // (epochs * eval_times_per_epoch)
# # Find the largest divisor of save_steps that is <= ideal_eval_frequency
# for divisor in range(ideal_eval_frequency, 0, -1):
#     if save_steps % divisor == 0:
#         eval_steps = divisor
#         break

# Calculated Steps Summary
print("  Calculated Steps:")
print(f"    Total Steps: {total_steps}")
print(f"    Warmup Steps: {warmup_steps}")
print(f"    Save Steps: {save_steps}")
print(f"    Eval Steps: {eval_steps}")
print(f"    Every N Steps: {every_n_steps}")
print("-" * 50)

  Calculated Steps:
    Total Steps: 126
    Warmup Steps: 12
    Save Steps: 63
    Eval Steps: 63
    Every N Steps: 6
--------------------------------------------------


In [102]:
save_steps % eval_steps

3

In [92]:
total_steps/eval_steps

14.0

In [4]:
new_token[:5], training_data[0]

(['[0]', '[1]', '[2]', '[3]', '[4]'],
 {'text': '<klon8> พอได้ยินเสียงระฆังข้างหลัง<r>[a][w]เขา</r>\tเห็นผู้<r>[a][w]เฒ่า</r>ออกจากชะวาก<r>[a]ผา</r>\nดูสรรพางค์ร่างกายแก่ช<r>[a]รา</r>\tแต่ผิว<r>[a]หน้า</r>นั้นละม้ายคล้ายทา<r>[o][k]รก</r>\nทรงเสื้อโขมพัสตรานุ่งผ้า<r>[a][w]ขาว</r>\tผมนั้น<r>[a][w]ยาว</r>ย้อยสยายประปราย<r>[o][k]ปรก</r>\nถือไม้เท้าเนาวรัตน์พัดขน<r>[o][k]นก</r>\tทำเดิน<r>[o][k]งก</r>งันมาแล้วพา<r>[i]ที</r>\nว่าดูราสามีนางผี<r>[Ua]เสื้อ</r>\tเป็นหน่อ<r>[Ua]เนื้อ</r>กษัตริย์ชาติราช<r>[i]สีห์</r>\nอย่าเผาศพนางยักษ์ด้วยอัค<r>[i]คี</r>\tภัยจะ<r>[i]มี</r>ถึงกายให้วาย<r>[a][n]ปราณ</r>\n'})

Add new token to tokenizer and model with 2 apporaches
1. Mean initialize the weights of the new token
2. Prior knowlege from training data


In [5]:
old_tokenizer_vocab_size = len(tokenizer)
old_model_vocab_size = model.vocab_size
old_tokenizer_vocab_size, old_model_vocab_size

(128256, 128256)

In [6]:
# Approach 1: Mean initialize the weights of the new token

tokenizer.add_tokens(new_token)
model.resize_token_embeddings(len(tokenizer))

new_tokenizer_vocab_size = len(tokenizer)
new_model_vocab_size = model.vocab_size

new_tokenizer_vocab_size, new_model_vocab_size

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


(128334, 128334)

In [8]:
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"}
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    padding=True,
    return_tensors="pt"
)

outputs = model.generate(
                input_ids,
                max_new_tokens=512,
                eos_token_id=tokenizer.eos_token_id,
                do_sample=True,
                temperature=0.9,
                top_p=0.9,
                top_k=64
        )
print(tokenizer.decode(outputs[0], skip_special_tokens=False))

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is the capital of France?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The capital of France is Paris.<|eot_id|>


In [10]:
model.save_pretrained("llama3.2-typhoon2-1b-instruct-new-token")
tokenizer.save_pretrained("llama3.2-typhoon2-1b-instruct-new-token")

('llama3.2-typhoon2-1b-instruct-new-token/tokenizer_config.json',
 'llama3.2-typhoon2-1b-instruct-new-token/special_tokens_map.json',
 'llama3.2-typhoon2-1b-instruct-new-token/tokenizer.json')

In [3]:
model = AutoModelForCausalLM.from_pretrained("llama3.2-typhoon2-1b-instruct-new-token")
tokenizer = AutoTokenizer.from_pretrained("llama3.2-typhoon2-1b-instruct-new-token")

messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"}
]

input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    padding=True,
    return_tensors="pt"
)

outputs = model.generate(
                input_ids,
                max_new_tokens=512,
                eos_token_id=tokenizer.eos_token_id,
                do_sample=True,
                temperature=0.9,
                top_p=0.9,
                top_k=64
        )
print(tokenizer.decode(outputs[0], skip_special_tokens=False))


[2025-05-15 11:04:54,085] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to mps (auto detect)


W0515 11:04:54.266000 35433 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are a helpful assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

What is the capital of France?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The capital of France is Paris.<|eot_id|>


In [4]:
model.vocab_size

128334

In [14]:
training_data[0]

{'text': '<klon8> พอได้ยินเสียงระฆังข้างหลัง<r>[a][w]เขา</r>\tเห็นผู้<r>[a][w]เฒ่า</r>ออกจากชะวาก<r>[a]ผา</r>\nดูสรรพางค์ร่างกายแก่ช<r>[a]รา</r>\tแต่ผิว<r>[a]หน้า</r>นั้นละม้ายคล้ายทา<r>[o][k]รก</r>\nทรงเสื้อโขมพัสตรานุ่งผ้า<r>[a][w]ขาว</r>\tผมนั้น<r>[a][w]ยาว</r>ย้อยสยายประปราย<r>[o][k]ปรก</r>\nถือไม้เท้าเนาวรัตน์พัดขน<r>[o][k]นก</r>\tทำเดิน<r>[o][k]งก</r>งันมาแล้วพา<r>[i]ที</r>\nว่าดูราสามีนางผี<r>[Ua]เสื้อ</r>\tเป็นหน่อ<r>[Ua]เนื้อ</r>กษัตริย์ชาติราช<r>[i]สีห์</r>\nอย่าเผาศพนางยักษ์ด้วยอัค<r>[i]คี</r>\tภัยจะ<r>[i]มี</r>ถึงกายให้วาย<r>[a][n]ปราณ</r>\n'}

In [15]:
import torch
import gc
from collections import defaultdict
import re

def extract_phonetic_combinations(training_data, tokenizer, model_type="gemma|llama"):
    assert(model_type in ["gemma", "llama"])
    # Define regex pattern to match <r>[X][Y]text</r> patterns
    pattern = r'<r>(.+?)</r>'

    tag_dict = defaultdict(set)

    # Process each text in the training data
    for data in training_data:
        # Find all matches of the pattern in the text
        text = data["text"]
        matches = re.findall(pattern, text)

        for item in matches:
            # Extract all tags (e.g., [a], [w])
            tags = re.findall(r'\[.*?\]', item)
            # Extract the word by removing tags
            word = re.sub(r'\[.*?\]', '', item).strip()
            # Add the word to each tag's set
            for tag in tags:
                if "<r>" in word:
                    print(
                        f"Warning: <r> tag found in word '{word}'. Skipping this entry.")
                    continue
                tag_dict[tag].add(word)

    new_tag_dict = defaultdict(set)
    for key, value in tag_dict.items():
        for word in value:
            if model_type == "gemma":
                tokenized_words = tokenizer.tokenizer.encode(word, add_special_tokens=False)
            else:
                tokenized_words = tokenizer.encode(word, add_special_tokens=False)
            # tokenized_words = tokenizer.encode(word, add_special_tokens=False)
            if "<r>" in tokenized_words:
                print(f"Value: {value}")
                print(f"Tokenized word: {word}")
                print(f"Tokenized word: {tokenized_words}")
            for tokenized_word in tokenized_words:
                new_tag_dict[key].add(tokenized_word)

    return new_tag_dict

# Custom function for Gemma-3 token adding
def mean_surround_new_tag_token(model, tag_dict):
    # Calculate the mean of surrounding token embeddings
    embedding_matrix = model.get_input_embeddings().weight.clone()
    lm_head_matrix = model.get_output_embeddings().weight.clone()

    tag_embedding_dict = {}
    tag_lm_head_dict = {}
    
    tag_embedding = torch.zeros_like(embedding_matrix[0])
    tag_lm_head = torch.zeros_like(lm_head_matrix[0])

    for tag, ids_values in tag_dict.items():
        # properly accumulate all the embeddings
        tag_embedding.zero_()
        tag_lm_head.zero_()
        for idx in ids_values:
            tag_embedding += embedding_matrix[idx]
            tag_lm_head += lm_head_matrix[idx]
        tag_embedding_dict[tag] = tag_embedding / len(ids_values)
        tag_lm_head_dict[tag] = tag_lm_head / len(ids_values)

    return tag_embedding_dict, tag_lm_head_dict


def add_new_tokens(
    model,
    tokenizer,
    tag_dict,
    new_tokens=[],
    model_type="gemma|llama"
):
    """
    Smartly resizes the tokenizer and adds new tokens to the model.
    We also disregard untrained tokens by removing them from the mean calculation.
    """
    # All Unsloth Zoo code licensed under LGPLv3
    assert (isinstance(new_tokens, (list, tuple)))
    assert (len(new_tokens) > 0)
    assert (len(tag_dict) > 0)
    assert (isinstance(tag_dict, dict))
    assert(model_type in ["gemma", "llama"])

    # Check if tokens already exist
    if model_type == "gemma":
        overlapping_tokens = set(new_tokens) & set(tokenizer.tokenizer.vocab.keys())
    else:
        overlapping_tokens = set(new_tokens) & set(tokenizer.vocab.keys())
        
    if len(overlapping_tokens) != 0:
        print(
            f"Unsloth: You're adding new_tokens = {new_tokens}\n"
            f"There are tokens which are overlapping = {list(overlapping_tokens)}\n"
            f"We shall safely ignore these overlapping tokens."
        )
        new_tokens = [x for x in new_tokens if x not in overlapping_tokens]


    # Weirdly be careful reserved tokens can pop out
    tag_embedding_dict, tag_lm_head_dict = mean_surround_new_tag_token(
        model, tag_dict)

    # Get old lengths
    old_input_embedding = model.get_input_embeddings().weight
    old_output_embedding = model.get_output_embeddings().weight
    old_input_length = old_input_embedding.shape[0]
    old_output_length = old_output_embedding.shape[0]
    if model_type == "gemma":
        old_config_size = model.config.text_config.vocab_size
    else:
        old_config_size = model.config.vocab_size

    # Check for tied weights as well
    is_tied = (old_input_embedding.data_ptr() == old_output_embedding.data_ptr()) \
        or (model.config.tie_word_embeddings)

    # Add tokens!
    if model_type == "gemma":
        old_length = len(tokenizer.tokenizer)
        tokenizer.tokenizer.add_tokens(new_tokens)
        model.resize_token_embeddings(len(tokenizer.tokenizer))
    else:
        old_length = len(tokenizer)
        tokenizer.add_tokens(new_tokens)
        model.resize_token_embeddings(len(tokenizer))
    # Also resizes lm_head as well!

    # the Word2Vec sum of the other vectors
    embedding_matrix = model.get_input_embeddings().weight
    lm_head_matrix = model.get_output_embeddings().weight

    # Confirm sizes are correct
    if embedding_matrix.shape[0] > (old_input_length + len(new_tokens)):
        raise RuntimeError(
            "Unsloth: Embedding matrix size did not get resized properly. Please file a bug report!"
        )
    if lm_head_matrix.shape[0] > (old_output_length + len(new_tokens)):
        raise RuntimeError(
            "Unsloth: LM Head matrix size did not get resized properly. Please file a bug report!"
        )
    if model_type == "gemma":
        if model.config.text_config.vocab_size > (old_config_size + len(new_tokens)):
            raise RuntimeError(
                "Unsloth: Model's config vocab_size did not get resized properly. Please file a bug report!"
            )
    else:
        if model.config.vocab_size > (old_config_size + len(new_tokens)):
            raise RuntimeError(
                "Unsloth: Model's config vocab_size did not get resized properly. Please file a bug report!"
            )

    key_list = list(tag_dict.keys())
    if model_type == "gemma":
        key_ids_list = [tokenizer.tokenizer.encode(word, add_special_tokens=False)[0] for word in key_list]
    else:
        key_ids_list = [tokenizer.encode(word, add_special_tokens=False)[0] for word in key_list]
    with torch.no_grad():
        for key, ids in zip(key_list, key_ids_list):
            tag_embedding = tag_embedding_dict[key]
            tag_lm_head = tag_lm_head_dict[key]
            embedding_matrix[ids] = tag_embedding
            lm_head_matrix[ids] = tag_lm_head
            pass

    # We set a flag to say we need to train embeddings
    internal_model = model
    while hasattr(internal_model, "model"):
        internal_model._need_to_train_embeddings = True
        internal_model = internal_model.model
    pass
    internal_model._need_to_train_embeddings = True
    
    # Fix up all vocab sizes
    current_model = model
    if hasattr(current_model, "model") and hasattr(current_model, "config"):
        if model_type == "gemma":
            if hasattr(current_model.config.text_config, "vocab_size"):
                current_model.config.text_config.update({"vocab_size": len(tokenizer.tokenizer)})
        else:
            if hasattr(current_model.config, "vocab_size"):
                current_model.config.update({"vocab_size": len(tokenizer)})
        current_model = current_model.model

    # Must tie lm_head and embed_tokens if they are tied!
    # Otherwise error will occur on saving models ie use save_model
    if is_tied:
        model.tie_weights()

    # Clear deleted GPU items
    for _ in range(3):
        gc.collect()
        torch.cuda.empty_cache()
    return

In [16]:
tag_dict = extract_phonetic_combinations(training_data, tokenizer, model_type="llama")

In [23]:
add_new_tokens(model=model, tokenizer=tokenizer, tag_dict=tag_dict, new_tokens=new_token, model_type="llama")

Unsloth: You're adding new_tokens = ['[0]', '[1]', '[2]', '[3]', '[4]', '[?]', '[@@-]', '[@@]', '[@]', '[N]', '[OO-]', '[OO]', '[O]', '[UU]', '[UUa]', '[Ua]', '[U]', '[a-]', '[a]', '[aa]', '[aee]', '[b]', '[bl]', '[br]', '[c]', '[ch]', '[d]', '[dr]', '[e]', '[ee]', '[f]', '[fl]', '[fr]', '[h]', '[i]', '[ia]', '[ii-]', '[ii]', '[iia]', '[j]', '[k]', '[kh]', '[khl]', '[khr]', '[khw]', '[kl]', '[kr]', '[kw]', '[l]', '[m]', '[n]', '[o]', '[oo]', '[p]', '[ph]', '[phl]', '[phr]', '[pl]', '[pr]', '[r]', '[s]', '[sw]', '[t]', '[th]', '[thr]', '[tr]', '[u]', '[ua]', '[uu-]', '[uu]', '[uua]', '[w]', '[x]', '[xx-]', '[xx]', '<klon8>', '<r>', '</r>']
There are tokens which are overlapping = ['[oo]', '[ch]', '[kr]', '[khw]', '[aee]', '[k]', '[U]', '[d]', '[f]', '[ee]', '[uu-]', '[n]', '[o]', '[s]', '[a]', '[2]', '[xx-]', '[uu]', '[h]', '[phr]', '[kw]', '</r>', '<r>', '[3]', '[w]', '[p]', '[c]', '[b]', '[i]', '[th]', '[a-]', '[m]', '[iia]', '[r]', '[UUa]', '[j]', '[uua]', '[e]', '[l]', '[dr]', '[@@]

In [24]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model = AutoModelForCausalLM.from_pretrained("llama3.2-typhoon2-1b-smart-tokens")
tokenizer = AutoTokenizer.from_pretrained("llama3.2-typhoon2-1b-smart-tokens")

In [25]:
model.vocab_size


128334